**Trying to Implement Decoder-only GPT-style model using PyTorch.**
### Project Pipeline
- Dataset: **Kaggle 64K Recipe Dataset**
- Preprocessing: **Sentencepiece**
- Model: **Decoder Only GPT-style**
- Evaluation: **No**
- Max output length: *512*
- Approx Accuracy: **22% - 26%**


In [ ]:
import re
import torch
import torch.nn as nn
import torch.functional as F
import sentencepiece as sp
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from model import GPT, Config

In [ ]:
#building data for feeding
def format_recipe(recipe):
    return f"Title: {recipe['recipe_title']}\nCategory: {recipe['category']}\n Sub-Category:{recipe['subcategory']} \nDescription: {recipe['description']}\n Number of nIngredients:{recipe['num_ingredients']} \nIngredients: {recipe['ingredients']}\nDirections: {recipe['directions']}"

def clean_text(t):
    t = t.lower()
    t = re.sub(r'[^a-zA-Z0-9.,:/%()\- ]+', ' ', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()
    
df = pd.read_csv("dataset/1_Recipe_csv.csv")

df['recipe'] = df.apply(format_recipe, axis=1)
df["recipe"] = df["recipe"].apply(clean_text)
len(df["recipe"])

62126

In [6]:
df["recipe"].iloc[0]

'title: air fryer potato slices with dipping sauce category: air fryer recipes sub-category:air fryer recipes description: these air fryer potato slices, served with a beer ketchup dipping sauce, are a tasty finger food somewhere between a french fry and a potato chip. do take the time to make the dipping sauce it s worth it. number of ningredients:9 ingredients: 3/4 cup ketchup , 1/2 cup beer , 1 tablespoon worcestershire sauce , 1/2 teaspoon onion powder , 1/4 teaspoon cayenne , 2 baking potatoes , olive oil cooking spray , 1/2 teaspoon garlic powder , salt and freshly ground black pepper directions: combine ketchup, beer, worcestershire sauce, onion powder, and cayenne in a small saucepan. bring to a boil, then reduce heat, and simmer for 3 to 5 minutes. remove from heat, and cool. cover and store in the refrigerator until ready to use. , preheat the air fryer to 400 degrees f (200 degrees c). spray the basket with cooking spray or line with a disposable parchment liner. , slice pot

In [9]:
data = df['recipe'].astype(str).to_list()

with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in data:
        f.write(line.strip() + "\n")

In [11]:
sp.SentencePieceTrainer.Train(
    input="/kaggle/working/corpus.txt",
    model_prefix="tokenizer",
    vocab_size=8000,          
    character_coverage=1.0,   # 1 for English
    model_type="bpe",         # can also use "unigram"
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

In [12]:
sp_preprocessor = sp.SentencePieceProcessor()
sp_preprocessor.Load("/kaggle/working/tokenizer.model")

True

In [13]:
ids = sp_preprocessor.Encode("this is a sample", out_type=int)
print(ids)
print(sp_preprocessor.Decode(ids))

[244, 162, 7, 4489, 877]
this is a sample


In [14]:
encoded = [sp_preprocessor.Encode(t, out_type=int) for t in data]
flatten = torch.tensor([id for seq in encoded for id in seq], dtype=torch.long)

In [15]:
#dataset class
class MyDataset(Dataset):
    def __init__(self, encoded, seq_len):
        self.data = encoded
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)//self.seq_len
    
    def __getitem__(self, index):
        start = index * self.seq_len
        x = self.data[start : start + self.seq_len]
        y = self.data[start+1 : start + self.seq_len + 1]
        return x, y

In [17]:
dataset = MyDataset(flatten, seq_len=128)

#custom data loader 
def build_dataloader(ds, batch_size, num_workers=4):
    return DataLoader(
        ds, batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=4
        )

In [18]:
loader = build_dataloader(dataset, 16)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [19]:
#model initialization
model = GPT(config=Config())
model.to(device)

GPT(
  (token_embed): Embedding(50000, 512, padding_idx=0)
  (positional_embed): Embedding(512, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-7): 8 x GPTBlock(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=50000, bias=False)
)

In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    weight_decay=0.01
)
import math

In [29]:
def train_step(model, loader, optimizer, scaler, scheduler, device, grad_accum=2):
    model.train()
    total_loss = 0.0
    steps = 0
    criterion = torch.nn.CrossEntropyLoss()
    j = 0
    print("[", end="")
    for i, (x, y) in enumerate(loader):
        x = x.to(device)
        y = y.to(device)
        j = j + 1
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            logits = model(x)
            B, S, V = logits.shape

            logits = logits.reshape(B*S, V)
            y = y.reshape(B*S)

            loss = criterion(logits, y) / grad_accum

        # accumulate gradients
        scaler.scale(loss).backward()

        # Only unscale + clip + step when accumulation boundary reached
        if (i + 1) % grad_accum == 0:
            # unscale once
            scaler.unscale_(optimizer)

            # clip grad once
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # optimizer step
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            # schedule lr
            if scheduler:
                scheduler.step()

        total_loss += loss.item() * grad_accum
        steps += 1
        if j%45 == 0:
            print("=",end="")
    print("]",end="")
    return total_loss / steps


In [30]:
def train(model, loader, optimizer, device, epochs=6):
    scaler = torch.cuda.amp.GradScaler()

    warmup_steps = 1500
    total_steps = len(loader) * epochs

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / warmup_steps
        # cosine decay
        progress = float(step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        loss = train_step(
            model, loader,
            optimizer, scaler,
            scheduler, device
        )
        print(f"Loss: {loss:.4f}")


In [33]:
train(model, loader, optimizer, device, epochs=10)

Epoch 1/10
[=============================================================================================================================================================================================]Loss: 6.1348
Epoch 2/10
[=============================================================================================================================================================================================]Loss: 3.0505
Epoch 3/10
[=============================================================================================================================================================================================]Loss: 2.6083
Epoch 4/10
[=============================================================================================================================================================================================]Loss: 2.3565
Epoch 5/10
[================================================================================================================================

In [34]:
torch.save(model.state_dict(),"recipe_gpt_model.pth")

In [53]:
def sample_logits(logits, temperature=1.0, top_k=50):
    # logits: (vocab_size)
    logits = logits / temperature

    if top_k is not None and top_k > 0:
        v, ix = torch.topk(logits, top_k)
        logits2 = torch.full_like(logits, float('-inf'))
        logits2[ix] = v
        logits = logits2

    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    return next_id

In [54]:
@torch.no_grad()
def generate(model, input_ids, max_new_tokens=50, temperature=1.0, top_k=40):
    model.eval()
    device = next(model.parameters()).device

    input_ids = input_ids.to(device)

    for _ in range(max_new_tokens):
        # Forward pass
        logits = model(input_ids)          # (B, S, V)
        logits = logits[:, -1, :]          # last token logits (B, V)

        # Sample next token
        next_token = sample_logits(
            logits[0],
            temperature=temperature,
            top_k=top_k
        )

        # Append
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

        # Stop if EOS exists and is generated
        if next_token.item() == getattr(model.config, "eos_token_id", None):
            break

    return input_ids

In [55]:
def generate_text(model, sp, prompt, max_new_tokens=80, temperature=0.9, top_k=50):
    device = next(model.parameters()).device

    # Encode using sentencepiece
    input_ids = torch.tensor([sp.encode(prompt, out_type=int)], dtype=torch.long)

    # Generate
    out = generate(
        model,
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k
    )

    # Convert IDs → text
    ids = out[0].tolist()
    text = sp.decode(ids)
    return text

In [56]:
prompt = "how to make french fries"
text = generate_text(model, sp_preprocessor, prompt, max_new_tokens=20)
print(text)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
